### Loading the Fine-Tuned Sentiment Classifier

We begin the inference notebook by loading the BERT model and tokenizer fine-tuned in the training notebook. The HuggingFace pipeline abstraction is used to wrap the model into a convenient text-classification interface. This allows us to pass raw text and obtain sentiment predictions and confidence scores in a single call, which simplifies the later integration with summarization.

In [66]:
!pip install transformers --quiet 

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)

In [67]:
tokenizer = AutoTokenizer.from_pretrained("./sentiment_model")
model = AutoModelForSequenceClassification.from_pretrained("./sentiment_model")

sentiment_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=False
)


C:\Users\rsree\anaconda3\Lib\site-packages\transformers\pipelines\text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


### Loading NLTK product_reviews Dataset

#### Loading Unlabeled NLTK Product Reviews and Preprocessing
To evaluate how well the model generalizes beyond the training corpus, we use the product_reviews_1 dataset from NLTK, which contains reviews of several consumer products without sentiment labels. The review_to_text helper converts each NLTK review object into a single plain-text string by concatenating its sentences. We then apply a very light cleaning function that only collapses excessive whitespace; punctuation and casing are preserved because the BERT tokenizer is robust to such variation. The resulting nltk_clean_texts list provides unseen real-world input on which we can test the sentiment and summarization pipeline

In [70]:
import nltk
from nltk.corpus import product_reviews_1

# Download the corpus
nltk.download("product_reviews_1")

# Show list of product files
product_ids = product_reviews_1.fileids()
print("Number of products:", len(product_ids))
print("Sample product IDs:", product_ids[:5])


Number of products: 6
Sample product IDs: ['Apex_AD2600_Progressive_scan_DVD player.txt', 'Canon_G3.txt', 'Creative_Labs_Nomad_Jukebox_Zen_Xtra_40GB.txt', 'Nikon_coolpix_4300.txt', 'Nokia_6610.txt']


[nltk_data] Downloading package product_reviews_1 to
[nltk_data]     C:\Users\rsree\AppData\Roaming\nltk_data...
[nltk_data]   Package product_reviews_1 is already up-to-date!


#### Extract Reviews for One Product

In [72]:
# Choose the first product for demonstration
first_product = product_ids[0]
print("Using product:", first_product)

# Load reviews (Review objects)
nltk_reviews = product_reviews_1.reviews(first_product)
print("Number of reviews:", len(nltk_reviews))


Using product: Apex_AD2600_Progressive_scan_DVD player.txt
Number of reviews: 99


#### Convert NLTK Review Objects to Plain Text

In [74]:
def review_to_text(review):
    return " ".join([w for sent in review.sents() for w in sent])

# Converts all reviews
nltk_texts = [review_to_text(r) for r in nltk_reviews]

# example
print(nltk_texts[0][:300])


repost from january 13 , 2004 with a better fit title . does your apex dvd player only play dvd audio without video ? or does it play audio and video but scrolling in black and white ? before you try to return the player or waste hours calling apex tech support , or run the player over with your car


#### Light Clean the NLTK Reviews

In [76]:
import re

def clean_nltk_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)  # remove extra spaces
    return text.strip()

nltk_clean_texts = [clean_nltk_text(t) for t in nltk_texts]

print(nltk_clean_texts[0][:300])


repost from january 13 , 2004 with a better fit title . does your apex dvd player only play dvd audio without video ? or does it play audio and video but scrolling in black and white ? before you try to return the player or waste hours calling apex tech support , or run the player over with your car


### Predict Sentiment Using Your Fine-Tuned BERT
#### Sentiment Inference on Unseen Reviews
The predict_sentiment function wraps the pipeline into a convenient interface that returns a human-readable label (“positive” or “negative”) together with the model’s confidence. We apply this function to the first five NLTK reviews and print review snippets alongside the predictions. This experiment demonstrates that the fine-tuned BERT model can be seamlessly deployed to analyze new product reviews from a different source, highlighting its potential for real-world applications.

In [78]:
def predict_sentiment(text):
    """
    Runs sentiment analysis on a single review.
    Returns: label (positive/negative), confidence score
    """
    output = sentiment_classifier(text)[0]
    label_id = int(output["label"].split("_")[-1])
    sentiment = "negative" if label_id == 0 else "positive"
    confidence = float(output["score"])
    return sentiment, confidence


#### Test Sentiment on NLTK Reviews

In [80]:
for r in nltk_clean_texts[:5]:
    sentiment, score = predict_sentiment(r)
    print("REVIEW:", r[:200], "...\n")
    print("Predicted Sentiment:", sentiment, "| Confidence:", round(score, 3))
    print("-" * 80)


REVIEW: repost from january 13 , 2004 with a better fit title . does your apex dvd player only play dvd audio without video ? or does it play audio and video but scrolling in black and white ? before you try  ...

Predicted Sentiment: positive | Confidence: 0.656
--------------------------------------------------------------------------------
REVIEW: i ' ve owned 6 or 7 dvd players since 1998 . this is by far the nicest one , in so many ways . it ' s very sleek looking with a very good front panel button layout , and it has a great feature set . i ...

Predicted Sentiment: positive | Confidence: 0.723
--------------------------------------------------------------------------------
REVIEW: many of our disney movies do n ' t play on this dvd player . i spoke to a rep . at apex and was told that a new format is being used and we wont be able to view any disney movies made in the last coup ...

Predicted Sentiment: positive | Confidence: 0.688
----------------------------------------------

### Load BART Summarization Model

#### Abstractive Summarization and ROUGE Evaluation
To complement sentiment analysis with a concise textual description of each review, we adopt the pre-trained facebook/bart-large-cnn model. The summarize_review function tokenizes an input review, runs beam-search decoding, and returns a short abstractive summary. We first test the function on a single NLTK review to qualitatively inspect the output.

For a more systematic evaluation we compute ROUGE-1, ROUGE-2, and ROUGE-L scores on a set of 50 NLTK reviews. As the dataset does not provide human-written summaries, we construct simple heuristic reference summaries from the first one or two sentences of each review. The resulting ROUGE scores (approximately 0.69 for ROUGE-1 and 0.65 for ROUGE-2) indicate that BART captures much of the content of the original review and produces coherent, compressed representations of user feedback.

In [82]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

bart_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")


C:\Users\rsree\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


#### Summarize a Review

In [84]:
from transformers import BartTokenizer, BartForConditionalGeneration
import torch

# 1. Load model & tokenizer once
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

# 2. Define your function
def summarize_review(text, max_len=40):
    inputs = bart_tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    )
    
    bart_model.eval()
    with torch.no_grad():
        summary_ids = bart_model.generate(
            inputs["input_ids"],
            max_length=max_len,
            num_beams=4,
            early_stopping=True
        )
    
    return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# 3. call it
review_text = "This product is amazing! I have been using it for a month..."
summary = summarize_review(review_text)
print(summary)


C:\Users\rsree\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\rsree\anaconda3\Lib\site-packages\transformers\generation\utils.py:1165: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (40). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(


This product is amazing! I have been using it for a month... it's been amazing. I have used it for over a month and it has made a huge difference in my


#### Test Summarization

In [86]:
sample_review = nltk_clean_texts[0]

print("ORIGINAL REVIEW:")
print(sample_review[:400])

print("\nSUMMARY:")
print(summarize_review(sample_review))


ORIGINAL REVIEW:
repost from january 13 , 2004 with a better fit title . does your apex dvd player only play dvd audio without video ? or does it play audio and video but scrolling in black and white ? before you try to return the player or waste hours calling apex tech support , or run the player over with your car , try these simple troubleshooting ideas first . no picture : hopefully you still have the remote c

SUMMARY:


C:\Users\rsree\anaconda3\Lib\site-packages\transformers\generation\utils.py:1165: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (40). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(


i have done some research and experimenting with the remote when my apex ad - 2500 seemed to have lost its video signal. i hope i have helped you continue to enjoy the apex dvd


In [87]:
# ROUGE EVALUATION FOR BART SUMMARIES

# Installs rouge-score once in the environment
!pip install -q rouge-score

from rouge_score import rouge_scorer
import numpy as np

# ROUGE scorer for 3 common variants
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def naive_reference_summary(text, max_sentences=2):
    """
    Create a simple 'reference summary' by taking the first 1–2 sentences.
    This is a heuristic since we don't have gold summaries in NLTK.
    """
    sentences = text.split(". ")
    return ". ".join(sentences[:max_sentences])

# NLTK reviews for evaluation
eval_texts = nltk_clean_texts[:50]  

scores = {"rouge1": [], "rouge2": [], "rougeL": []}

for t in eval_texts:
    reference = naive_reference_summary(t)
    generated = summarize_review(t)

    rouge = scorer.score(reference, generated)
    for k in scores.keys():
        scores[k].append(rouge[k].fmeasure)

print("AVERAGE ROUGE SCORES ON NLTK SAMPLE")
for k in scores:
    print(f"{k}: {np.mean(scores[k]):.4f}")

AVERAGE ROUGE SCORES ON NLTK SAMPLE
rouge1: 0.6934
rouge2: 0.6549
rougeL: 0.6841


### Final Combined Function (Sentiment + Summary)

#### Integrated Sentiment and Summarization Pipeline
The analyze_review function ties the whole project together. Given a raw review, it first calls the fine-tuned BERT classifier to predict sentiment and confidence, and then passes the same text to the BART summarizer to generate a short abstractive summary. We apply this pipeline to several NLTK reviews and present the results in a table containing truncated original text, predicted sentiment, confidence score, and model summary. This final experiment demonstrates a complete end-to-end system for automatically analyzing large collections of product reviews, suitable for integration into an e-commerce analytics dashboard or decision-support tool.

In [89]:
def analyze_review(text):
    sentiment, score = predict_sentiment(text)
    summary = summarize_review(text)
    
    return {
        "review": text,
        "sentiment": sentiment,
        "confidence": round(score, 3),
        "summary": summary
    }

final_result = analyze_review(nltk_clean_texts[0])
final_result


C:\Users\rsree\anaconda3\Lib\site-packages\transformers\generation\utils.py:1165: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (40). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(


{'review': "repost from january 13 , 2004 with a better fit title . does your apex dvd player only play dvd audio without video ? or does it play audio and video but scrolling in black and white ? before you try to return the player or waste hours calling apex tech support , or run the player over with your car , try these simple troubleshooting ideas first . no picture : hopefully you still have the remote control . if you tossed it out the window , you need to fetch it . using the remote control , press the i / p button located on the bottom right corner of the remote . the i / p button switches the tv display between interlace and progressive . if this doesnt bring back the picture , try pressing this button without playing a dvd . if you dont get video back , now you can run the player over with your car ! picture scrolling in b / w : you need the remote control for this so you better get it from your dog before he burries it in the backyard . press the p / n button located on the 

In [90]:
import pandas as pd  

examples = []

for r in nltk_clean_texts[:5]:
    result = analyze_review(r)
    examples.append({
        "Original review (truncated)": result["review"][:200] + "...",
        "Predicted sentiment": result["sentiment"],
        "Confidence": result["confidence"],
        "Summary": result["summary"]
    })

examples_df = pd.DataFrame(examples)
examples_df


,Original review (truncated),Predicted sentiment,Confidence,Summary
0,"repost from january 13 , 2004 with a better fi...",positive,0.656,i have done some research and experimenting wi...
1,i ' ve owned 6 or 7 dvd players since 1998 . t...,positive,0.723,i've owned 6 or 7 dvd players since 1998. this...
2,many of our disney movies do n ' t play on thi...,positive,0.688,many of our disney movies don't play on this d...
3,player has a problem with dual - layer dvd ' s...,positive,0.711,player has a problem with dual - layer dvd's s...
4,"for the first few weeks , this player was ever...",positive,0.550,The apex 2600 is actually ruining my media. Th...
